<a href="https://colab.research.google.com/github/suhaasteja/VOD_reviewer/blob/main/VOD_review.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Clash Royale VOD Reviewer 👑

Clash Royale is a fast-paced competitive mobile game where players deploy troops
to destroy the opponent's towers — whoever captures the most towers before the
clock runs out wins.

**What's a VOD?**
VOD stands for *Video on Demand* — in gaming, it refers to a recorded replay of
a match. Reviewing your own VODs is one of the most effective ways to improve:
you see your decisions from the outside, spot patterns you missed in the moment,
and identify exactly where a game was won or lost. Pro players and coaches do
this constantly. The problem is it's time-consuming and requires a trained eye.

**Where Perceptron comes in**
This notebook uses Perceptron Mk1 — a vision model that understands video — to
automate that review process. Instead of scrubbing through footage manually, you
ask a question in plain English and get a grounded, timestamped answer backed by
what the model actually saw on screen. No transcripts, no audio analysis, no
game-specific training data — pure visual reasoning on the gameplay footage itself.

The final 30 seconds are where matches are decided. That's the window we focus on.

**What it does:**
- **VOD analysis** — reviews the last 30s of a match and diagnoses the single
  defining play: the outplay that secured the win, or the mistake that cost the loss
- **Card tracking** — upload a reference image of any troop card and the model
  will track how many times it was deployed and the damage it dealt (powered by
  in-context learning — no retraining required)

In [ ]:
%pip install --upgrade perceptron --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.3/65.3 kB 2.3 MB/s eta 0:00:00


## Step 1 — Load the gameplay clip

We download a sample 30-second Clash Royale clip from the repo directly into
the Colab runtime. To use your own footage, replace `VIDEO_URL` with a link to
your video or upload a file and update `VIDEO_PATH` accordingly.

> **Note:** Perceptron Mk1 processes video at 2fps and caps at ~128 seconds per
> API call. A 30-second clip is ideal — it fits in a single call with no
> preprocessing required.

In [ ]:
VIDEO_URL = "https://raw.githubusercontent.com/suhaasteja/VOD_reviewer/main/content/clash-royale_last_30s.mp4"
### make sure your video is < 20mb
!curl -L -so last_30_s.mp4 {VIDEO_URL}

### Preview — final 30 seconds

The clip below is the window Mk1 will analyze. Watch it once before running
the analysis — it helps you validate the model's output against what you saw.

<video src="https://raw.githubusercontent.com/suhaasteja/VOD_reviewer/main/content/clash-royale_last_30s.mp4" controls width="200" muted loop>
  Your browser doesn't support the video tag.
</video>

## Step 2 — Configure Perceptron

Set your API key in Colab's **Secrets** panel (the 🔑 icon in the left sidebar)
under the name `PERCEPTRON_API_KEY`. The cell below reads it securely and
initialises the Mk1 client.

Don't have a key yet? Get one at [perceptron.inc](https://platform.perceptron.inc/).

In [ ]:
import os
from google.colab import userdata

from perceptron import configure, question, video, image, perceive, text


api_key = userdata.get('PERCEPTRON_API_KEY')
if not api_key or api_key.startswith("<"):
    raise RuntimeError("Set PERCEPTRON_API_KEY or replace the placeholder in this cell.")

configure(
    provider="perceptron",
    model="perceptron-mk1",
    api_key=api_key,
)

VIDEO_PATH = "last_30_s.mp4"

## Step 3 — VOD analysis

The prompt below is the core of the review. It instructs Mk1 to:
1. Determine the match outcome from the visual game state (winner screen, tower counts)
2. Branch its analysis — diagnosing the decisive mistake on a loss, or surfacing
   hidden inefficiencies on a win

You can customise this prompt for any game or analysis style. The model reads
the gameplay visually — no audio, no OCR workaround, no game-specific training.

In [ ]:
VOD_PROMPT = """Analyze these final 30 seconds of Clash Royale gameplay. Evaluate my performance based on whether this clip is a win or a loss:

- IF A LOSS: Diagnose the single defining mistake that cost me the match. Evaluate my win-condition targeting, elixir/cycle management, and card placement. Suggest the exact alternative card sequence that would have secured a win or tie.
- IF A WIN: Identify the key outplay or macro decision that secured the victory. Spot any hidden inefficiencies, resource mismanagement, or structural vulnerabilities I still committed despite winning.

"""

In [ ]:
QUESTION = VOD_PROMPT

result = question(video(VIDEO_PATH), QUESTION, reasoning=True, expects="clip")

print("--- Reasoning ---")
print(result.reasoning or "(none)")
print("\n--- Answer ---")
print(result.text)

--- Reasoning ---
So, let's analyze this Clash Royale clip. First, check if it's a win or loss. The video shows "Match Over!" and then the winner screen with "Suhass" as the winner (since the blue side is Suhass, and the crown is on their side). So it's a win.

Now, identify the key outplay or macro decision. Let's look at the gameplay:

- The player (Suhass) is on the blue side. The opponent (juao) is red.
- In the final seconds, the opponent's king tower is still up, but the player's towers are holding. Wait, no—wait, the match ends with the player winning, so the opponent's towers must have been destroyed.

Looking at the timeline: The timer counts down, and at the end, "Match Over!" appears, then the winner screen shows Suhass as winner. So the player won.

Key outplay: Let's see the card plays. The player uses a combination of troops and spells. For example, in the later part, maybe a well-timed spell or troop placement that prevented the opponent from pushing.

Wait, the opponent

## Step 4 — Card-level tracking with in-context learning

Standard vision models need to be retrained to recognise new objects. Mk1
doesn't — you can show it a reference image at query time and it will track
that object throughout the video.

Here we upload a single screenshot of the **Hog Rider** card and ask Mk1 to
count deployments and measure tower damage across the clip. Swap in any card
image to track a different troop — no code changes needed.

This is Perceptron's **in-context learning** capability: the reference image
is passed as part of the prompt, not baked into the model.

In [ ]:
EXAMPLE_IMAGE = "https://raw.githubusercontent.com/suhaasteja/VOD_reviewer/main/content/hog-rider-on-field.jpg"

!curl -L -so hog-rider-on-field.jpg {EXAMPLE_IMAGE}

In [ ]:
EXAMPLE_IMAGE="hog-rider-on-field.jpg"
QUERY_VIDEO="last_30_s.mp4"

### Reference image — Hog Rider

This is the image we pass to Mk1 as the visual example. The model uses it to
identify the Hog Rider in the gameplay footage, even though it has never been
explicitly trained on Clash Royale card art.

<img src="https://raw.githubusercontent.com/suhaasteja/VOD_reviewer/main/content/hog-rider-on-field.jpg" width="100" alt="Hog Rider">

In [ ]:
@perceive(reasoning=True, expects="clip")
def check_card_info(example_image_path: str, query_video_path: str):
    return (
        image(example_image_path)
        + text("this card is called hog-rider from a game called clash royale")
        + video(query_video_path)
        + text("how many times was it played and how much opponents tower health amount did it reduce?")
    )

result = check_card_info(EXAMPLE_IMAGE, QUERY_VIDEO)

print("--- Reasoning ---")
print(result.reasoning or "(none)")
print("\n--- Answer ---")
print(result.text)

--- Reasoning ---
So, let's look at the video. First, the Hog Rider is played once in the gameplay. Now, check the opponent's tower health. The opponent's tower (the one with the red crown) starts with some health. Let's see the numbers. In the video, when the Hog Rider attacks, the tower's health goes from 1274 to 843. Wait, let's calculate the reduction: 1274 - 843 = 431. So the Hog Rider was played once, and it reduced the opponent's tower health by 431.

--- Answer ---
To determine how many times the Hog Rider was played and the health reduction on the opponent’s tower:  

1. **Number of times played**: In the video, the Hog Rider is deployed *once* during the match (visible as the character is placed on the battlefield).  

2. **Opponent’s tower health reduction**:  
   - The opponent’s tower (top - red - crowned structure) starts with a health value of **1274** (seen early in the gameplay).  
   - After the Hog Rider’s attack, the tower’s health drops to **843** (visible later in

todo
- generate clips for multiple cards
- generate structured output for analytics
- use LLM as tool / use mk1 as a tool (read docs)

## Inspect the returned clips

In [ ]:
clips = result.clips or []
if not clips:
    print("No clips returned. Try rephrasing the question or pointing at a different moment.")
else:
    print(f"Returned {len(clips)} clip(s):")
    for idx, clip in enumerate(clips, start=1):
        ts = clip.timestamp
        if ts.until is None:
            window = f"moment at {ts.at:.2f}s"
        else:
            window = f"{ts.at:.2f}s - {ts.until:.2f}s"
        label = clip.mention or "(no mention)"
        print(f"  Clip {idx}: {window} - {label}")

No clips returned. Try rephrasing the question or pointing at a different moment.
